# Getting Started with voiage

A short first-run notebook showing the main stable workflow.

In [ ]:
import numpy as np
from voiage.analysis import DecisionAnalysis
from voiage.schema import ValueArray, ParameterSet, TrialDesign, DecisionOption
from voiage.methods.sample_information import evsi, enbs

values = np.array([
    [10.0, 11.5],
    [9.8, 11.2],
    [10.3, 11.1],
    [9.7, 11.4],
])
value_array = ValueArray.from_numpy(values, ['standard care', 'new treatment'])
analysis = DecisionAnalysis(nb_array=value_array)
print(f'EVPI: {analysis.evpi():.4f}')


In [ ]:
parameters = ParameterSet.from_numpy_or_dict({
    'treatment_effect': np.array([0.2, 0.3, 0.4, 0.5]),
    'cost_shift': np.array([0.0, 0.1, 0.0, -0.1]),
})
analysis_with_params = DecisionAnalysis(nb_array=value_array, parameter_samples=parameters)
print(f'EVPPI: {analysis_with_params.evppi(n_regression_samples=4):.4f}')


In [ ]:
trial_design = TrialDesign(arms=[
    DecisionOption(name='New Treatment', sample_size=10),
    DecisionOption(name='Standard Care', sample_size=10),
])

def simple_model(psa):
    treatment = psa.parameters['treatment_effect']
    cost_shift = psa.parameters['cost_shift']
    base = np.column_stack([
        100.0 + 2.0 * cost_shift,
        101.0 + 4.0 * treatment - cost_shift,
    ])
    return ValueArray.from_numpy(base, ['standard', 'new'])

psa_prior = ParameterSet.from_numpy_or_dict({
    'treatment_effect': np.linspace(0.1, 0.5, 12),
    'cost_shift': np.linspace(-0.2, 0.2, 12),
})
print(f'EVSI: {evsi(simple_model, psa_prior, trial_design, method="efficient"):.4f}')


## Exercise

Change the `new treatment` column so the EVPI gets smaller, then re-run the calculation.

In [ ]:
improved = ValueArray.from_numpy(
    np.array([
        [20000.0, 24000.0],
        [21000.0, 23900.0],
        [20500.0, 24100.0],
    ]),
    ["Standard care", "New treatment"],
)
print(f'Exercise solution EVPI: {DecisionAnalysis(nb_array=improved).evpi():.2f}')